# 4장 - GPT Vision 이미지 분석

원본 파일: `chap04/vision_basic.py`

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import base64

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [2]:
def encode_image(image_path: str) -> str:
    """로컬 이미지 파일을 base64 문자열로 변환 (URL이 없는 내 컴퓨터 사진을 GPT에 보낼 때 사용)"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [3]:
def describe_image_by_url(image_url: str, question: str = "이 이미지에 대해 설명해주세요.") -> str:
    """인터넷에 있는 이미지(URL)를 GPT에게 설명해달라고 요청"""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {
                    "type": "image_url",
                    "image_url": {"url": image_url},
                },
            ],
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o",
        messages=messages,
    )
    return response.choices[0].message.content

In [4]:
def _to_data_url(image_path: str) -> str:
    """확장자에 맞는 MIME 타입을 붙여서 data URL 문자열을 만듦 (jpg/png 둘 다 지원)"""
    mime = "image/png" if image_path.lower().endswith(".png") else "image/jpeg"
    return f"data:{mime};base64,{encode_image(image_path)}"

In [5]:
def describe_image_by_path(image_path: str, question: str = "이 이미지에 대해 설명해주세요.") -> str:
    """내 컴퓨터에 있는 로컬 이미지 파일을 GPT에게 설명해달라고 요청 (base64로 인코딩해서 전달)"""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": _to_data_url(image_path)}},
            ],
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o",
        messages=messages,
    )
    return response.choices[0].message.content

In [6]:
def compare_images(image_path_1: str, image_path_2: str, question: str) -> str:
    """로컬 이미지 두 장을 함께 보여주고 비교 질문을 요청"""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": _to_data_url(image_path_1)}},
                {"type": "image_url", "image_url": {"url": _to_data_url(image_path_2)}},
            ],
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-4o",
        messages=messages,
    )
    return response.choices[0].message.content

## 실행 / 테스트

In [7]:
IMAGES_DIR = os.path.join((os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()), 'data', 'images')

In [8]:
print("===== ① URL 이미지 설명 =====")
image_url = "https://images.unsplash.com/photo-1736264335247-8ec5664c8328?q=80&w=1887&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
print(describe_image_by_url(image_url))

===== ① URL 이미지 설명 =====
이미지에는 한 여성이 주방에서 음식 준비를 하는 모습이 보입니다. 그녀는 운동복을 입고 있으며, 파란색 그릇에 계란을 휘젓고 있습니다. 그녀는 미소를 짓고 있어 즐거운 시간을 보내고 있는 것처럼 보입니다. 주위에는 조리대와 조리 중인 다른 음식들도 보입니다.


In [9]:
print("\n===== ② 로컬 이미지 설명 (망원동 베이커리) =====")
print(describe_image_by_path(os.path.join(IMAGES_DIR, "mangwon_bakery.jpg")))


===== ② 로컬 이미지 설명 (망원동 베이커리) =====
이 이미지는 베이커리의 진열대를 보여주고 있습니다. 진열대에는 다양한 종류의 빵이 가지런히 놓여 있으며, 각 빵에는 이름과 가격이 적혀 있습니다. 배경에는 직원으로 보이는 사람들이 일하고 있고, 진열된 빵은 주로 바게트, 크루아상, 단팥빵, 머핀 등 다양한 종류가 포함되어 있습니다. 진열대는 유리로 되어 있어 빵이 잘 보입니다.


In [10]:
print("\n===== ③ 두 카페 비교 (선릉 테라로사 vs 로컬스티치 테라로사) =====")
print(compare_images(
    os.path.join(IMAGES_DIR, "seolleung_terrarosa.jpg"),
    os.path.join(IMAGES_DIR, "local_stitch_terrarosa.jpg"),
    "두 카페의 차이점을 설명해주세요.",
))


===== ③ 두 카페 비교 (선릉 테라로사 vs 로컬스티치 테라로사) =====
첫 번째 카페와 두 번째 카페의 차이점은 다음과 같습니다:

1. **디자인과 분위기**:
   - 첫 번째 카페는 어두운 색조의 인테리어와 목재 바닥이 돋보이며, 차분하고 조용한 분위기를 제공합니다.
   - 두 번째 카페는 밝은 색상과 노란색의 포인트 벽이 있어 현대적이고 활기찬 분위기를 느낄 수 있습니다.

2. **조명과 창문**:
   - 첫 번째 카페는 자연광이 창문을 통해 들어오며, 따뜻하고 아늑한 조명을 사용했습니다.
   - 두 번째 카페는 더 밝은 인공 조명에 의해 전체 공간이 환하게 비춰지며 넓은 창문에 의해 외부와 소통하는 느낌입니다.

3. **가구 배치 및 스타일**:
   - 첫 번째 카페는 비교적 널찍하게 테이블이 배치된 반면, 두 번째 카페는 테이블이 더욱 조직적으로 배열되어 있습니다.
   - 가구의 스타일도 첫 번째 카페는 클래식한 느낌이며, 두 번째 카페는 현대적이고 미니멀한 디자인을 채택하고 있습니다.

두 카페 모두 각자의 개성과 스타일을 가지고 있어 다양한 취향을 만족시킬 수 있을 것입니다.


In [11]:
print("\n===== ④ GPT Vision 한계 테스트 (OECD 연구개발비 그래프, 해상도별 비교) =====")
question = "첫번째는 2021년 데이터이고, 두번째는 2022년 데이터입니다. 이 데이터에 대해 설명해주세요. 어떤 변화가 있었나요?"


===== ④ GPT Vision 한계 테스트 (OECD 연구개발비 그래프, 해상도별 비교) =====


In [12]:
print("--- large 해상도 ---")
print(compare_images(
    os.path.join(IMAGES_DIR, "oecd_rnd_2021_large.png"),
    os.path.join(IMAGES_DIR, "oecd_rnd_2022.png"),
    question,
))

--- large 해상도 ---
두 개의 이미지에서 2021년과 2022년 각국의 연구개발비와 GDP 대비 연구개발비 비중 변화를 비교해 보겠습니다.

1. **연구개발비 (억만 USD) 변화**:
   - **미국**: 2021년 720,880에서 2022년 806,013으로 증가.
   - **중국**: 353,484에서 433,500으로 증가.
   - **일본**: 약간 감소했지만 거의 변동 없음.
   - **한국**: 89,282에서 91,013으로 소폭 증가.
   - **프랑스**, **독일** 등 대부분의 국가에서 연구개발비가 증가하거나 비슷한 수준.

2. **GDP 대비 연구개발비 비중 (%) 변화**:
   - **이스라엘**: 5.44%에서 5.56%로 증가.
   - **한국**: 4.93%에서 4.21%로 감소.
   - **스웨덴**: 3.49%에서 3.40%로 감소.
   - **대만**: 추가되며 2022년 3.77%로 기록됨.
   - 몇몇 국가에서는 소폭 변동 있었으나, 다른 대부분의 국가에서는 비슷한 수준 유지.

전반적으로, 여러 국가에서 연구개발비는 증가하는 경향을 보였으며, GDP 대비 비중은 국가마다 다소간의 변동이 있었습니다. 그러나 대부분의 국가에서 큰 변화 없이 유지되었습니다.


In [13]:
print("\n--- medium(낮은) 해상도 ---")
print(compare_images(
    os.path.join(IMAGES_DIR, "oecd_rnd_2021_medium.png"),
    os.path.join(IMAGES_DIR, "oecd_rnd_2022.png"),
    question,
))


--- medium(낮은) 해상도 ---
두 그래프는 각각 2021년과 2022년의 연구개발비와 GDP 대비 연구개발비 비중을 보여줍니다. 몇 가지 주요 변화를 정리하면 다음과 같습니다:

1. **미국**: 연구개발비가 720,880백만 달러에서 806,013백만 달러로 증가했습니다. GDP 대비 비율은 3.45%에서 3.46%로 약간 높아졌습니다.

2. **중국**: 연구개발비가 353,484백만 달러에서 433,500백만 달러로 증가했습니다. 비율은 2.40%에서 2.43%로 증가했습니다.

3. **일본, 독일**: 이들 국가들도 연구개발비는 소폭 변동했지만, GDP 대비 비율에서는 큰 변화를 보이지 않았습니다.

4. **한국**: 연구개발비가 91,934백만 달러에서 87,225백만 달러로 감소했고, GDP 대비 비율도 3.15%에서 2.91%로 감소했습니다.

5. **프랑스**: 연구개발비가 69,289백만 달러에서 65,641백만 달러로 감소했고, GDP 대비 비율은 여전히 2.22%로 유지되었습니다.

이처럼 두 해 간 연구개발비와 GDP 대비 비율에서 몇 가지 주요 변동 사항이 있었음을 알 수 있습니다. 전체적으로 대부분의 국가에서 연구개발비가 증가하거나 유지되었지만, 한국과 프랑스에서는 감소했습니다.
